In [0]:
%run ../00-common/01.environment-config

In [0]:
# SOURCE AND TARGET PATHS

source_path = f"{s3_root_path}/Bronze/Aaisha/results"
target_path = f"{s3_root_path}/Silver/Aaisha/results"

print("Source Path :", source_path)
print("Target Path :", target_path)

In [0]:
results_df = (
    spark.read
        .format("delta")
        .load(source_path)
)

display(results_df)

In [0]:
from pyspark.sql import functions as F

results_clean_df = (
    results_df
        .dropna(how="all")
        .dropDuplicates()
        .filter(F.col("driverId").isNotNull())
        .filter(F.col("constructorId").isNotNull())
        .filter(F.col("season").isNotNull())
        .filter(F.col("round").isNotNull())
        .withColumn("constructorId", F.trim(F.col("constructorId")))
        .withColumn("driverId", F.trim(F.col("driverId")))
        .withColumn("raceName", F.trim(F.col("raceName")))
        .withColumn("status", F.trim(F.col("status")))
        .withColumn("positionText", F.trim(F.col("positionText")))
        .withColumn("url", F.trim(F.col("url")))
)

In [0]:
results_final_df = results_clean_df.select(
    "season",
    "round",
    "raceName",
    "driverId",
    "constructorId",
    "number",
    "grid",
    "position",
    "positionText",
    "points",
    "laps",
    "status",
    "date",
    "url",
    "ingestion_timestamp",
    "source_file"
)

display(results_final_df)

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
silver_results_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(silver_results_df)